In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("merged_electronics_dataset.csv")

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 200)

print("Rows, Columns:", df.shape)
display(df.head())


Rows, Columns: (5010, 10)

In [ ]:
# show column list and dtypes
print(df.columns.tolist())
print(df.dtypes)

# a quick summary of missing values and basic stats
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing>0].head(20))
display(df.describe(include='all').T)


['name', 'main_category', 'sub_category', 'image', 'link', 'no_of_ratings', 'discount_price', 'actual_price', 'review_rating', 'review_text']
name              object
main_category     object
sub_category      object
image             object
link              object
no_of_ratings     object
discount_price    object
actual_price      object
review_rating     object
review_text       object
dtype: object

discount_price    270
no_of_ratings      34
actual_price       31
review_text         3
dtype: int64

In [ ]:
# typical target column names: 'price' or 'selling_price' — find which exists
for c in ['price','selling_price','price_usd','sellingprice']:
    if c in df.columns:
        print("Found target column:", c)
        target = c
        break
else:
    print("No standard target column found — check dataset for price field.")
    target = None

# examine unique categories and brands
for col in ['category','brand','product_category','title']:
    if col in df.columns:
        print(col, "— unique count:", df[col].nunique())
        display(df[col].value_counts().head(10))


No standard target column found — check dataset for price field.

In [ ]:
    # create a working sample
sample = df.sample(frac=0.1, random_state=42)  # 10% sample
sample.to_csv("amazon_electronics_sample.csv", index=False)
print("Saved sample (10%) to amazon_electronics_sample.csv")


Saved sample (10%) to amazon_electronics_sample.csv

In [ ]:
import numpy as np
import re

# 1️⃣ Drop useless columns
df = df.drop(['image', 'link', 'main_category', 'sub_category', 'review_text'], axis=1)

# 2️⃣ Clean numeric-looking columns
def extract_number(text):
    if pd.isna(text): 
        return np.nan
    # remove currency symbol, commas, text like '₹', 'FREE', etc.
    text = re.sub(r'[^\d.]', '', str(text))
    try:
        return float(text)
    except:
        return np.nan

df['discount_price'] = df['discount_price'].apply(extract_number)
df['actual_price'] = df['actual_price'].apply(extract_number)
df['no_of_ratings'] = df['no_of_ratings'].apply(extract_number)

# 3️⃣ Extract rating value (e.g., "4.2 out of 5 stars" → 4.2)
df['review_rating'] = df['review_rating'].str.extract(r'(\d+\.?\d*)').astype(float)

# 4️⃣ Handle missing values — drop rows where price is missing
df = df.dropna(subset=['discount_price', 'actual_price'])
df.fillna(0, inplace=True)

# 5️⃣ Create additional feature: discount percentage
df['discount_percent'] = ((df['actual_price'] - df['discount_price']) / df['actual_price']) * 100
df['discount_percent'] = df['discount_percent'].clip(lower=0, upper=100)  # avoid negative/outliers

# 6️⃣ Keep only relevant columns for modeling
df_model = df[['name', 'no_of_ratings', 'discount_price', 'actual_price', 'review_rating', 'discount_percent']]

print("✅ Cleaned Data Sample:")
display(df_model.head())
print("Data Types:\n", df_model.dtypes)

✅ Cleaned Data Sample:

Data Types:
 name                 object
no_of_ratings       float64
discount_price      float64
actual_price        float64
review_rating       float64
discount_percent    float64
dtype: object

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1️⃣ Feature / target separation
X = df_model[['actual_price', 'review_rating', 'no_of_ratings', 'discount_percent']]
y = df_model['discount_price']

# 2️⃣ Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

# 3️⃣ Scaling numeric features for better model performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# 1️⃣ Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

# 2️⃣ Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)  # RandomForest handles raw scale fine
rf_pred = rf.predict(X_test)


# 4️⃣ Evaluation function
def evaluate_model(y_true, y_pred, model_name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"📊 {model_name}")
    print(f"R² Score : {r2:.4f}")
    print(f"MAE      : {mae:.2f}")
    print(f"RMSE     : {rmse:.2f}\n")
    return r2, mae, rmse

lr_metrics = evaluate_model(y_test, lr_pred, "Linear Regression")
rf_metrics = evaluate_model(y_test, rf_pred, "Random Forest")

# 5️⃣ Plot: Actual vs Predicted for best model (Random Forest)
plt.figure(figsize=(6,4))
plt.scatter(y_test, rf_pred, alpha=0.6, color='teal')
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted Price (Random Forest)")
plt.show()

📊 Linear Regression
R² Score : 0.9277
MAE      : 1038.62
RMSE     : 2321.08

📊 Random Forest
R² Score : 0.9963
MAE      : 83.58
RMSE     : 521.64

<Figure size 600x400 with 1 Axes>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1️⃣ Get feature importance from the trained Random Forest
importance = pd.Series(rf.feature_importances_, index=X.columns)

# 2️⃣ Sort and visualize
importance.sort_values(ascending=True).plot(kind='barh', color='mediumseagreen', figsize=(6,4))
plt.title("Feature Importance — Factors Affecting Price")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

# 3️⃣ Print ranked values
print("Feature Importance Scores:\n", importance.sort_values(ascending=False))

<Figure size 600x400 with 1 Axes>

Feature Importance Scores:
 actual_price        0.927821
discount_percent    0.067759
no_of_ratings       0.003934
review_rating       0.000486
dtype: float64